In [24]:
import sqlite3
import pandas as pd
from pathlib import Path

clean_dir = Path("../data/clean")
db_path = Path("../data/olist.db")

conn = sqlite3.connect(db_path)


In [25]:
tables = ["orders", "products", "order_items", "zip_coordinates", "customers", "sellers"]

read_options = {                                         
    "orders": {
        "parse_dates": ["order_purchase_timestamp", "order_approved_at",
                        "order_delivered_carrier_date", "order_delivered_customer_date",
                        "order_estimated_delivery_date"],
        "dtype": {"is_delayed": "boolean"},
    },
    "order_items": {"parse_dates": ["shipping_limit_date"]},
}

data = {}

for name in tables:
   df = pd.read_csv(clean_dir / f"{name}.csv", **read_options.get(name, {}))
   data[name] = df
   df.to_sql(name, conn, if_exists="replace", index = False)
   print(f"{name}: {len(df)} rows saved")
 

orders: 99441 rows saved
products: 32951 rows saved
order_items: 112650 rows saved
zip_coordinates: 19015 rows saved
customers: 99441 rows saved
sellers: 3095 rows saved


In [26]:
pd.read_sql("SELECT name FROM sqlite_master WHERE type = 'table'", conn)  

,name
0,orders
1,products
2,order_items
3,zip_coordinates
4,customers
5,sellers


In [27]:
for name in tables:
    n = pd.read_sql(f"SELECT count(*) AS n FROM {name}", conn)["n"].iloc[0]
    print(f"{name}: {n}")
  

orders: 99441
products: 32951
order_items: 112650
zip_coordinates: 19015
customers: 99441
sellers: 3095


In [28]:
is_delayed = pd.read_sql("SELECT COUNT(*) FROM orders WHERE is_delayed IS NULL", conn)
is_delayed

,COUNT(*)
0,2965


In [29]:
average = pd.read_sql("SELECT AVG(is_delayed) FROM orders", conn)
average

,AVG(is_delayed)
0,0.067737


In [30]:

indexes_config = {
    "orders": ["order_id", "customer_id"],
    "products": ["product_id"],
    "customers": ["customer_id", "customer_zip_code_prefix"],
    "sellers": ["seller_id"],
    "zip_coordinates": ["geolocation_zip_code_prefix"],
    "order_items": ["order_id", "product_id", "seller_id"],
}

for table, columns in indexes_config.items():
    for col in columns:
        conn.execute(
            f"CREATE INDEX IF NOT EXISTS idx_{table}_{col} ON {table} ({col})"
        )

conn.commit()

pd.read_sql("SELECT name, tbl_name FROM sqlite_master WHERE type = 'index'", conn)

,name,tbl_name
0,idx_orders_order_id,orders
1,idx_orders_customer_id,orders
2,idx_products_product_id,products
3,idx_customers_customer_id,customers
4,idx_customers_customer_zip_code_prefix,customers
5,idx_sellers_seller_id,sellers
6,idx_zip_coordinates_geolocation_zip_code_prefix,zip_coordinates
7,idx_order_items_order_id,order_items
8,idx_order_items_product_id,order_items
9,idx_order_items_seller_id,order_items


In [31]:
items_value = pd.read_sql("SELECT SUM(items_value) FROM orders", conn)
items_value

orders = data["orders"]

total = orders["items_value"].sum()

print(f"{items_value} | {total}")

   SUM(items_value)
0        13591643.7 | 13591643.7


In [32]:
pd.read_sql("SELECT strftime('%Y-%m', order_purchase_timestamp) FROM orders", conn).head()
pd.read_sql("SELECT count(*) FROM orders WHERE strftime('%Y-%m', order_purchase_timestamp) IS NULL", conn)

,count(*)
0,0


In [33]:
pd.read_sql("SELECT count(*) FROM orders o LEFT JOIN customers c ON o.customer_id = c.customer_id WHERE c.customer_id IS NULL", conn)


,count(*)
0,0


In [43]:
pd.read_sql("SELECT DISTINCT is_delayed FROM orders",conn)
#data["orders"]
#data["products"]
data["order_items"]
#conn.close()  

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93
2,000229ec398224ef6ca0657da4fc703e,1,c777355d18b72b67abbeef9df44fd0fd,5b51032eddd242adc84c38acab88f23d,2018-01-18 14:48:30,199.00,17.87
3,00024acbcdf0a6daa1e931b038114c75,1,7634da152a4610f1595efa32f14722fc,9d7a1d34a5052409006425275ba1c2b4,2018-08-15 10:10:18,12.99,12.79
4,00042b26cf59d7ce69dfabb4e55b4fd9,1,ac6c3623068f30de03045865e4e10089,df560393f3a51e74553ab94004ba5c87,2017-02-13 13:57:51,199.90,18.14
...,...,...,...,...,...,...,...
112645,fffc94f6ce00a00581880bf54a75a037,1,4aa6014eceb682077f9dc4bffebc05b0,b8bc237ba3788b23da09c0f1f3a3288c,2018-05-02 04:11:01,299.99,43.41
112646,fffcd46ef2263f404302a634eb57f7eb,1,32e07fd915822b0765e448c4dd74c828,f3c38ab652836d21de61fb8314b69182,2018-07-20 04:31:48,350.00,36.53
112647,fffce4705a9662cd70adb13d4a31832d,1,72a30483855e2eafc67aee5dc2560482,c3cfdc648177fdbbbb35635a37472c53,2017-10-30 17:14:25,99.90,16.95
112648,fffe18544ffabc95dfada21779c9644f,1,9c422a519119dcad7575db5af1ba540e,2b3e4a2a3ea8e01938cabda2a3e5cc79,2017-08-21 00:04:32,55.99,8.72


The six clean tables were loaded into a SQLite database (data/olist.db) with to_sql, using if_exists="replace" so that running the notebook again gives the same result instead of duplicating rows. SQLite has no date type: dates are stored as text, booleans as 0/1, and missing values as NULL.

I created indexes on every column used to join tables (named idx_table_column).